# EEGMAT EEGNet - 8-channel PGD-CW evaluation with KD deployment export

This notebook evaluates EEGNet on EEGMAT with grouped leave-subject-out cross-validation: all 36 subjects are still held out exactly once, but they are batched into 6 held-out subject groups instead of 36 single-subject folds. Each EDF is trimmed to the 8 selected 10-20 channels F3, F4, Fz, P3, P4, Pz, O1, and O2 before epoching/alignment. The notebook can also save fold-level adversarial teacher checkpoints, distill them into a smaller student, and export the student for deployment.

Each grouped LOSO fold does the following:

1. Train a clean EEGNet, selecting the checkpoint by validation balanced accuracy.
2. Continue from that clean checkpoint with plain untargeted PGD adversarial training under an L-infinity threat model.
3. Evaluate both the clean model and the PGD-adversarially trained model on the held-out subject group using clean metrics and PGD-CW-Linf robust metrics across an epsilon sweep.

Training PGD still follows the plain cross-entropy PGD recipe from Chen, Wang, and Wu (2024): initialize with uniform random noise in [-epsilon, epsilon], ascend cross-entropy with `sign(grad)`, and project after every step back into the L-infinity epsilon ball around the benign EEG epoch. Validation and final robust evaluation now use the untargeted CW margin objective, `best_other_logit - true_class_logit`, for PGD ascent and best-restart selection. Restarts are vectorized by expanding the restart dimension into the batch dimension.

Results are aggregated across the 6 grouped held-out folds as mean +/- 95% CI per epsilon per model. The paired Wilcoxon test compares clean vs. PGD-adversarially trained robust balanced accuracy at the main training epsilon using one paired value per held-out group.

**Compute cost warning.** This is still heavier than clean-only LOSO: 6 folds x (clean training + PGD adversarial training + multi-epsilon PGD-CW evaluation on two models). Use the epoch counts, PGD steps, distillation epochs, and epsilon list near the bottom as starting points for a full run or shorten them for a smoke test.


In [ ]:
!wget -r -N -c -np https://physionet.org/files/eegmat/1.0.0/ -P ./eegmat
!pip install mne --quiet


In [ ]:
import os
import random
from pathlib import Path
from collections import defaultdict

import mne
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from mne.io import read_raw_edf
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, f1_score, cohen_kappa_score
from scipy import stats
from scipy.stats import wilcoxon

TARGET_SR = 256
EPOCH_LEN_SEC = 2
SELECTED_CHANNELS = ("F3", "F4", "Fz", "P3", "P4", "Pz", "O1", "O2")
C = len(SELECTED_CHANNELS)  # selected EEGMAT channels
T = 512         # 2s @ 256Hz
SEED = 42
DATA_DIR = "./eegmat/physionet.org/files/eegmat/1.0.0"


In [ ]:
def _canonical_channel_name(name):
    name = str(name).strip().upper()
    name = name.replace("EEG", "").replace(" ", "").replace(".", "")
    name = name.split("-")[0]
    return name


def _selected_channel_indices(raw, selected_channels=SELECTED_CHANNELS):
    lookup = {_canonical_channel_name(ch_name): i for i, ch_name in enumerate(raw.ch_names)}
    requested = [_canonical_channel_name(ch) for ch in selected_channels]
    missing = [ch for ch, key in zip(selected_channels, requested) if key not in lookup]
    if missing:
        available = ", ".join(raw.ch_names)
        raise ValueError(
            f"Missing requested EEG channel(s) {missing} in {raw.filenames[0]}. "
            f"Available channels: {available}"
        )
    return [lookup[key] for key in requested]


def load_eegmat_subject(filepath, epoch_len_sec=EPOCH_LEN_SEC, sfreq_target=TARGET_SR,
                        selected_channels=SELECTED_CHANNELS):
    """Load one EDF and return epochs shaped (trials, selected channels, time)."""
    raw = read_raw_edf(filepath, preload=True, verbose=False)
    pick_indices = _selected_channel_indices(raw, selected_channels)
    raw.resample(sfreq_target, verbose=False)
    data = raw.get_data(picks=pick_indices).astype(np.float32, copy=False)

    samples_per_epoch = int(round(epoch_len_sec * raw.info["sfreq"]))
    n_epochs = data.shape[1] // samples_per_epoch
    if n_epochs == 0:
        raise ValueError(f"{filepath} is shorter than one {epoch_len_sec:g}-second epoch")

    trimmed = data[:, : n_epochs * samples_per_epoch]
    epochs = trimmed.reshape(data.shape[0], n_epochs, samples_per_epoch).transpose(1, 0, 2)
    return np.ascontiguousarray(epochs)


def load_eegmat_dataset(data_dir, subject_ids):
    """Per-subject rest epochs and task epochs, kept SEPARATE so the EA
    reference can be fit on rest only (see euclidean_alignment_causal)."""
    rest_by_subj, task_by_subj = {}, {}
    for subj in subject_ids:
        rest_file = os.path.join(data_dir, f"Subject{subj:02d}_1.edf")
        task_file = os.path.join(data_dir, f"Subject{subj:02d}_2.edf")
        rest_by_subj[subj] = load_eegmat_subject(rest_file)
        task_by_subj[subj] = load_eegmat_subject(task_file)
    return rest_by_subj, task_by_subj


def _fit_whitener(X, eigenvalue_floor=1e-12):
    X = np.asarray(X)
    reference = np.einsum("nct,ndt->cd", X, X, dtype=np.float64, optimize=True) / X.shape[0]
    reference = 0.5 * (reference + reference.T)
    eigvals, eigvecs = np.linalg.eigh(reference)
    inv_sqrt = np.clip(eigvals, eigenvalue_floor, None) ** -0.5
    return (eigvecs * inv_sqrt) @ eigvecs.T


def euclidean_alignment_causal(rest_epochs, task_epochs):
    """Fit EA on the rest block only (available before the task block in real
    time); apply that fixed whitener to both rest and task epochs. No task
    (test) trial contributes to its own transform, and the reference is not
    a blend of both classes."""
    whitener = _fit_whitener(rest_epochs).astype(np.float32)
    rest_aligned = whitener @ rest_epochs
    task_aligned = whitener @ task_epochs
    return np.ascontiguousarray(rest_aligned), np.ascontiguousarray(task_aligned), whitener


rest_by_subj, task_by_subj = load_eegmat_dataset(DATA_DIR, range(36))

aligned_data, subject_labels, alignment_whiteners = {}, {}, {}
for subj in rest_by_subj:
    rest_aligned, task_aligned, whitener = euclidean_alignment_causal(rest_by_subj[subj], task_by_subj[subj])
    alignment_whiteners[subj] = whitener
    aligned_data[subj] = np.concatenate([rest_aligned, task_aligned], axis=0)
    subject_labels[subj] = np.concatenate([
        np.zeros(rest_aligned.shape[0], dtype=np.int64),
        np.ones(task_aligned.shape[0], dtype=np.int64),
    ])

print("Subjects loaded:", len(aligned_data))


In [ ]:
class EEGMATAlignedDataset(Dataset):
    def __init__(self, X, y, subject_ids, normalize=True):
        X = np.ascontiguousarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.int64)
        subject_ids = np.asarray(subject_ids, dtype=np.int64)

        # Per-trial z-score across time only. This uses no information from
        # other trials or subjects, so it carries no leakage risk.
        if normalize:
            mu = X.mean(axis=-1, keepdims=True)
            sigma = X.std(axis=-1, keepdims=True)
            X = (X - mu) / np.maximum(sigma, 1e-6)

        self.X = torch.from_numpy(np.ascontiguousarray(X)).unsqueeze(1)
        self.y = torch.from_numpy(y)
        self.subject_ids = torch.from_numpy(subject_ids)

    def __len__(self):
        return int(self.y.numel())

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.subject_ids[idx]


def build_split(ids, aligned_data, subject_labels):
    arrays = [aligned_data[s] for s in ids]
    labels = [subject_labels[s] for s in ids]
    sids = [np.full(a.shape[0], s, dtype=np.int64) for s, a in zip(ids, arrays)]
    return np.concatenate(arrays, 0), np.concatenate(labels, 0), np.concatenate(sids, 0)


class EEGNET(nn.Module):
    def __init__(self, num_classes=2, C=21, T=512, F1=8, D=2, F2=16, dropout=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, T // 2), padding=(0, T // 4), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, kernel_size=(C, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout),
        )
        self._to_linear = self._get_flat_size(C, T)
        self.classifier = nn.Linear(self._to_linear, num_classes)

    def _get_flat_size(self, C, T):
        with torch.no_grad():
            x = self.block2(self.block1(torch.zeros(1, 1, C, T)))
        return x.flatten(1).shape[1]

    def forward(self, x):
        return self.classifier(self.block2(self.block1(x)).flatten(1))


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total = 0.0, 0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total += x.size(0)
    return total_loss / total


def update_confusion_matrix(pred, y, confusion):
    num_classes = confusion.size(0)
    encoded = y.to(torch.long) * num_classes + pred.to(torch.long)
    confusion += torch.bincount(
        encoded, minlength=num_classes * num_classes
    ).reshape(num_classes, num_classes)


def classification_metrics_from_confusion(confusion):
    confusion = confusion.float()
    total = confusion.sum()
    if total.item() == 0:
        return 0.0, 0.0, 0.0

    diag = confusion.diag()
    true_totals = confusion.sum(dim=1)
    pred_totals = confusion.sum(dim=0)

    acc = diag.sum() / total
    present = true_totals > 0
    recalls = torch.zeros_like(diag)
    recalls[present] = diag[present] / true_totals[present]
    bal_acc = recalls[present].mean() if present.any().item() else diag.new_tensor(0.0)

    f1_denom = true_totals + pred_totals
    f1_valid = f1_denom > 0
    f1_per_class = torch.zeros_like(diag)
    f1_per_class[f1_valid] = 2.0 * diag[f1_valid] / f1_denom[f1_valid]
    f1 = f1_per_class[f1_valid].mean() if f1_valid.any().item() else diag.new_tensor(0.0)
    return acc.item(), bal_acc.item(), f1.item()


@torch.inference_mode()
def evaluate_balanced(model, loader, device, num_classes=2):
    was_training = model.training
    model.eval()
    confusion = torch.zeros((num_classes, num_classes), dtype=torch.long, device=device)
    try:
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            update_confusion_matrix(pred, y, confusion)
        return classification_metrics_from_confusion(confusion)
    finally:
        model.train(was_training)


## Plain PGD training and PGD-CW evaluation components


In [ ]:
def get_logits(output):
    return output[0] if isinstance(output, tuple) else output


def linf_per_sample(delta):
    return delta.flatten(1).abs().amax(dim=1)


def true_and_best_logits(logits, labels):
    correct_logits = logits.gather(1, labels[:, None]).squeeze(1)
    true_class_mask = F.one_hot(labels, num_classes=logits.size(1)).bool()
    best_other_logits = logits.masked_fill(true_class_mask, -torch.inf).amax(dim=1)
    return correct_logits, best_other_logits


def cw_margin_per_sample(logits, labels, kappa=0.0):
    correct_logits, best_other_logits = true_and_best_logits(logits, labels)
    return best_other_logits - correct_logits + kappa


def cw_margin_loss(logits, labels, kappa=0.0):
    return cw_margin_per_sample(logits, labels, kappa=kappa).mean()


def attack_scores_per_sample(logits, labels, objective="ce", kappa=0.0):
    objective = objective.lower()
    if objective == "ce":
        return F.cross_entropy(logits, labels, reduction="none")
    if objective == "cw":
        return cw_margin_per_sample(logits, labels, kappa=kappa)
    raise ValueError(f"Unsupported PGD objective: {objective}")


def attack_objective(logits, labels, objective="ce", kappa=0.0):
    return attack_scores_per_sample(
        logits, labels, objective=objective, kappa=kappa
    ).mean()


def project_linf(delta, eps, detach=True):
    projected = delta.clamp(-eps, eps)
    return projected.detach() if detach else projected


class PGDLinfAttack:
    """Untargeted PGD-Linf with vectorized restarts and CE or CW objective."""

    def __init__(self, eps=0.01, n_steps=20, step_size=None,
                 random_init=True, n_restarts=1, objective="ce", kappa=0.0):
        self.eps = float(eps)
        self.n_steps = int(n_steps)
        self.step_size = float(self.eps / 10 if step_size is None else step_size)
        self.random_init = bool(random_init)
        self.n_restarts = max(1, int(n_restarts))
        self.objective = objective.lower()
        self.kappa = float(kappa)
        if self.objective not in {"ce", "cw"}:
            raise ValueError(f"Unsupported PGD objective: {objective}")
        self.threat_model = f"PGD-{self.objective.upper()}-Linf"

    def perturb(self, model, x, y):
        was_training = model.training
        model.eval()
        batch_size = x.size(0)
        restarts = self.n_restarts if self.random_init else 1

        try:
            x_repeated = x.unsqueeze(0).expand(restarts, *x.shape).reshape(
                restarts * batch_size, *x.shape[1:]
            )
            y_repeated = y.unsqueeze(0).expand(restarts, batch_size).reshape(-1)

            if self.random_init and self.eps > 0:
                delta = torch.empty_like(x_repeated).uniform_(-self.eps, self.eps)
            else:
                delta = torch.zeros_like(x_repeated)
            delta = project_linf(delta, eps=self.eps, detach=True)

            for _ in range(self.n_steps):
                delta.requires_grad_(True)
                logits = get_logits(model(x_repeated + delta))
                attack_loss = attack_objective(
                    logits, y_repeated, objective=self.objective, kappa=self.kappa
                )
                grad = torch.autograd.grad(attack_loss, delta, only_inputs=True)[0]
                with torch.no_grad():
                    delta = delta + self.step_size * grad.sign()
                    delta = project_linf(delta, eps=self.eps, detach=True)

            with torch.no_grad():
                logits = get_logits(model(x_repeated + delta))
                scores = attack_scores_per_sample(
                    logits, y_repeated, objective=self.objective, kappa=self.kappa
                ).reshape(restarts, batch_size)
                delta_by_restart = delta.reshape(restarts, batch_size, *x.shape[1:])
                best_restart = scores.argmax(dim=0)
                batch_idx = torch.arange(batch_size, device=x.device)
                best_delta = delta_by_restart[best_restart, batch_idx].contiguous()
            return best_delta.detach(), (x + best_delta).detach()
        finally:
            model.train(was_training)


def update_balanced_stats(pred, y, correct_per_class, total_per_class):
    num_classes = total_per_class.numel()
    total_per_class += torch.bincount(y, minlength=num_classes)
    correct_per_class += torch.bincount(y[pred.eq(y)], minlength=num_classes)


def compute_balanced_accuracy(correct_per_class, total_per_class):
    present = total_per_class > 0
    if not bool(present.any()):
        return 0.0
    recalls = correct_per_class[present].float() / total_per_class[present]
    return recalls.mean().item()


@torch.inference_mode()
def evaluate_clean_balanced(model, loader, device, num_classes=2):
    was_training = model.training
    model.eval()
    correct = torch.zeros(num_classes, dtype=torch.long, device=device)
    total = torch.zeros_like(correct)
    try:
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            update_balanced_stats(get_logits(model(x)).argmax(dim=1), y, correct, total)
        return compute_balanced_accuracy(correct, total)
    finally:
        model.train(was_training)


def evaluate_adversarial_balanced(model, loader, attacker, device, num_classes=2):
    was_training = model.training
    model.eval()
    correct = torch.zeros(num_classes, dtype=torch.long, device=device)
    total = torch.zeros_like(correct)
    try:
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            _, x_adv = attacker.perturb(model, x, y)
            with torch.no_grad():
                pred_adv = get_logits(model(x_adv)).argmax(dim=1)
            update_balanced_stats(pred_adv, y, correct, total)
        return compute_balanced_accuracy(correct, total)
    finally:
        model.train(was_training)


def pgd_adversarial_train_one_epoch_balanced(model, loader, optimizer, attacker, device,
                                             criterion=None, num_classes=2):
    """Plain PGD adversarial training: generate PGD examples, then train on them."""
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    model.train()
    total_loss, total = 0.0, 0
    adv_correct = torch.zeros(num_classes, dtype=torch.long, device=device)
    adv_total = torch.zeros_like(adv_correct)

    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        _, x_adv = attacker.perturb(model, x, y)

        optimizer.zero_grad(set_to_none=True)
        logits_adv = get_logits(model(x_adv))
        loss = criterion(logits_adv, y)
        loss.backward()
        optimizer.step()

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total += batch_size

        with torch.no_grad():
            update_balanced_stats(logits_adv.argmax(1), y, adv_correct, adv_total)

    return {
        "loss": total_loss / total,
        "adv_bal_acc": compute_balanced_accuracy(adv_correct, adv_total),
    }


def compute_fidelity_torch(x, delta, eps=1e-12):
    signal_norm = torch.linalg.vector_norm(x.flatten(1), dim=1)
    noise_norm = torch.linalg.vector_norm(delta.flatten(1), dim=1).clamp_min(eps)
    snr = 20.0 * torch.log10(signal_norm.clamp_min(eps) / noise_norm)
    return snr, linf_per_sample(delta)


def _cat_numpy(tensor_batches, dtype=np.float32):
    if not tensor_batches:
        return np.empty(0, dtype=dtype)
    return torch.cat(tensor_batches).numpy()


def evaluate_full(model, loader, device, attacker=None, compute_fidelity=True):
    was_training = model.training
    model.eval()
    all_labels, all_preds_clean, all_preds_adv = [], [], []
    snr_batches, linf_batches = [], []
    try:
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                preds_clean = get_logits(model(x)).argmax(dim=1)
            if attacker is not None:
                delta, x_adv = attacker.perturb(model, x, y)
                with torch.no_grad():
                    preds_adv = get_logits(model(x_adv)).argmax(dim=1)
                    if compute_fidelity:
                        snr, linf_vals = compute_fidelity_torch(x, delta)
                        snr_batches.append(snr.cpu())
                        linf_batches.append(linf_vals.cpu())
            else:
                preds_adv = preds_clean
            all_labels.append(y.cpu())
            all_preds_clean.append(preds_clean.cpu())
            all_preds_adv.append(preds_adv.cpu())
        return {
            "labels": _cat_numpy(all_labels, dtype=np.int64),
            "preds_clean": _cat_numpy(all_preds_clean, dtype=np.int64),
            "preds_adv": _cat_numpy(all_preds_adv, dtype=np.int64),
            "snr_vals": _cat_numpy(snr_batches),
            "linf_vals": _cat_numpy(linf_batches),
        }
    finally:
        model.train(was_training)


def compute_metrics(results):
    y = results["labels"]
    yhat_c = results["preds_clean"]
    yhat_a = results["preds_adv"]
    return {
        "A_clean": float((yhat_c == y).mean()),
        "A_rob": float((yhat_a == y).mean()),
        "bACC_clean": balanced_accuracy_score(y, yhat_c),
        "bACC_rob": balanced_accuracy_score(y, yhat_a),
        "F1_rob": f1_score(y, yhat_a, average="macro", zero_division=0),
        "kappa_rob": cohen_kappa_score(y, yhat_a),
    }


def make_train_linf_attacker(eps):
    return PGDLinfAttack(eps=eps, n_steps=10, step_size=0.01,
                         random_init=True, n_restarts=2)


def make_eval_linf_attacker(eps):
    return PGDLinfAttack(eps=eps, n_steps=50, step_size=eps / 10,
                         random_init=True, n_restarts=5, objective="cw", kappa=0.0)


## Checkpointing, Distillation, And Deployment Helpers

These helpers save the adversarially trained fold model with the selected channel list and causal-EA whitening matrices, then optionally distill that teacher into a smaller EEGNet student. The student can be saved as a regular PyTorch checkpoint and exported as TorchScript for deployment.


In [ ]:
def default_teacher_kwargs():
    return {"num_classes": 2, "C": C, "T": T, "F1": 8, "D": 2, "F2": 16, "dropout": 0.5}


def default_student_kwargs():
    return {"num_classes": 2, "C": C, "T": T, "F1": 4, "D": 1, "F2": 8, "dropout": 0.25}


def build_eegnet_from_kwargs(model_kwargs):
    return EEGNET(**dict(model_kwargs))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def distillation_loss(student_logits, teacher_logits, labels, temperature=4.0, alpha=0.7):
    temperature = float(temperature)
    alpha = float(alpha)
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / temperature, dim=1),
        F.softmax(teacher_logits / temperature, dim=1),
        reduction="batchmean",
    ) * (temperature ** 2)
    hard_loss = F.cross_entropy(student_logits, labels)
    return alpha * soft_loss + (1.0 - alpha) * hard_loss, hard_loss, soft_loss


def distill_student_one_epoch(student, teacher, loader, optimizer, device,
                              temperature=4.0, alpha=0.7, adv_attacker=None,
                              num_classes=2):
    teacher.eval()
    student.train()
    total_loss, total_hard, total_soft, total = 0.0, 0.0, 0.0, 0
    confusion = torch.zeros((num_classes, num_classes), dtype=torch.long, device=device)

    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        if adv_attacker is not None:
            _, x_used = adv_attacker.perturb(teacher, x, y)
        else:
            x_used = x

        with torch.no_grad():
            teacher_logits = get_logits(teacher(x_used))

        optimizer.zero_grad(set_to_none=True)
        student_logits = get_logits(student(x_used))
        loss, hard_loss, soft_loss = distillation_loss(
            student_logits, teacher_logits, y,
            temperature=temperature, alpha=alpha,
        )
        loss.backward()
        optimizer.step()

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total_hard += hard_loss.item() * batch_size
        total_soft += soft_loss.item() * batch_size
        total += batch_size

        with torch.no_grad():
            update_confusion_matrix(student_logits.argmax(1), y, confusion)

    _, bal_acc, f1 = classification_metrics_from_confusion(confusion)
    return {
        "loss": total_loss / total,
        "hard_loss": total_hard / total,
        "soft_loss": total_soft / total,
        "bal_acc": bal_acc,
        "f1": f1,
    }


def distill_student_from_teacher(teacher, train_loader, val_loader, device,
                                 student_kwargs=None, epochs=20, lr=5e-4,
                                 weight_decay=1e-4, temperature=4.0,
                                 alpha=0.7, adv_attacker=None):
    student_kwargs = default_student_kwargs() if student_kwargs is None else dict(student_kwargs)
    student = build_eegnet_from_kwargs(student_kwargs).to(device)
    optimizer = torch.optim.Adam(student.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))

    best_state = {k: v.detach().clone() for k, v in student.state_dict().items()}
    best_val_bal_acc = -1.0
    history = []

    for epoch in range(1, epochs + 1):
        train_stats = distill_student_one_epoch(
            student, teacher, train_loader, optimizer, device,
            temperature=temperature, alpha=alpha, adv_attacker=adv_attacker,
        )
        _, val_bal_acc, val_f1 = evaluate_balanced(student, val_loader, device)
        scheduler.step()
        row = {
            "epoch": epoch,
            **train_stats,
            "val_bal_acc": val_bal_acc,
            "val_f1": val_f1,
        }
        history.append(row)
        if val_bal_acc > best_val_bal_acc:
            best_val_bal_acc = val_bal_acc
            best_state = {k: v.detach().clone() for k, v in student.state_dict().items()}

    student.load_state_dict(best_state)
    student.eval()
    return student, history, best_val_bal_acc


def _state_dict_cpu(model):
    return {k: v.detach().cpu() for k, v in model.state_dict().items()}


def _alignment_whitener_subset(alignment_whiteners, subject_ids):
    if alignment_whiteners is None:
        return {}
    return {
        int(s): torch.as_tensor(alignment_whiteners[int(s)]).detach().cpu()
        for s in subject_ids
        if int(s) in alignment_whiteners
    }


def make_fold_artifact_name(fold_seed_offset, test_ids, model_name):
    subject_part = "-".join(f"{int(s):02d}" for s in test_ids)
    return f"fold{fold_seed_offset + 1:02d}_test-{subject_part}_{model_name}"


def save_deployment_checkpoint(path, model, model_name, model_kwargs,
                               train_subjects, val_subjects, test_subjects,
                               metrics=None, alignment_whiteners=None,
                               distillation_history=None, extra=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    all_fold_subjects = list(train_subjects) + list(val_subjects) + list(test_subjects)
    checkpoint = {
        "model_name": model_name,
        "model_class": "EEGNET",
        "model_kwargs": dict(model_kwargs),
        "model_state_dict": _state_dict_cpu(model),
        "selected_channels": tuple(SELECTED_CHANNELS),
        "input_shape": (1, C, T),
        "target_sr": TARGET_SR,
        "epoch_len_sec": EPOCH_LEN_SEC,
        "preprocessing": {
            "channel_order": tuple(SELECTED_CHANNELS),
            "alignment": "causal_euclidean_alignment_fit_on_rest",
            "normalization": "per_trial_time_zscore_across_time",
        },
        "alignment_whiteners_by_subject": _alignment_whitener_subset(
            alignment_whiteners, all_fold_subjects
        ),
        "fold": {
            "train_subjects": [int(s) for s in train_subjects],
            "val_subjects": [int(s) for s in val_subjects],
            "test_subjects": [int(s) for s in test_subjects],
        },
        "metrics": metrics or {},
        "distillation_history": distillation_history,
        "extra": extra or {},
    }
    torch.save(checkpoint, path)
    return str(path)


def export_torchscript_model(model, path, C=C, T=T):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    was_training = model.training
    original_device = next(model.parameters()).device
    model_cpu = model.to("cpu").eval()
    with torch.no_grad():
        example = torch.zeros(1, 1, C, T)
        traced = torch.jit.trace(model_cpu, example)
    traced.save(str(path))
    model.to(original_device)
    model.train(was_training)
    return str(path)


def load_deployment_checkpoint(path, map_location="cpu"):
    checkpoint = torch.load(path, map_location=map_location)
    model = build_eegnet_from_kwargs(checkpoint["model_kwargs"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint


def prepare_deployment_tensor(epochs_ct, whitener=None, normalize=True):
    """Convert one or more selected-channel epochs to model input.

    epochs_ct can be shaped (C, T) for one epoch or (N, C, T) for a batch.
    If a causal-EA whitener is provided, it is applied before per-trial
    time-axis z-scoring, matching EEGMATAlignedDataset.
    """
    X = np.asarray(epochs_ct, dtype=np.float32)
    if X.ndim == 2:
        X = X[None, ...]
    if X.ndim != 3:
        raise ValueError("Expected epochs shaped (C, T) or (N, C, T)")
    if whitener is not None:
        W = np.asarray(whitener, dtype=np.float32)
        X = np.einsum("cd,ndt->nct", W, X, optimize=True)
    if normalize:
        mu = X.mean(axis=-1, keepdims=True)
        sigma = X.std(axis=-1, keepdims=True)
        X = (X - mu) / np.maximum(sigma, 1e-6)
    return torch.from_numpy(np.ascontiguousarray(X)).unsqueeze(1)


## Per-fold grouped LOSO function

`run_loso_fold_full` reuses the clean-training logic from the baseline notebook, then continues training the same checkpoint with plain PGD-Linf adversarial training. It evaluates both the clean checkpoint and the PGD-adversarially trained checkpoint on one held-out subject group per fold, across an epsilon sweep, using a PGD-CW-Linf attack for robust validation and final evaluation.


In [ ]:
def _as_subject_list(subjects):
    if isinstance(subjects, (int, np.integer)):
        return [int(subjects)]
    return [int(s) for s in subjects]


def make_subject_folds(subject_ids, n_folds=6, seed=SEED, shuffle=True):
    """Split subjects into grouped held-out folds.

    With 36 EEGMAT subjects and n_folds=6, each fold holds out 6 subjects.
    Every subject appears in exactly one held-out fold.
    """
    subject_ids = [int(s) for s in subject_ids]
    if not 1 <= n_folds <= len(subject_ids):
        raise ValueError("n_folds must be between 1 and the number of subjects")

    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(subject_ids)

    folds = [sorted(int(s) for s in fold) for fold in np.array_split(subject_ids, n_folds)]
    covered = sorted(s for fold in folds for s in fold)
    if covered != sorted(subject_ids):
        raise RuntimeError("Grouped folds must cover every subject exactly once")
    return folds


def run_loso_fold_full(test_subj, all_subj_ids, aligned_data, subject_labels, device,
                        n_val_subjects=4,
                        clean_epochs=30, clean_lr=1e-4, clean_weight_decay=1e-3,
                        pgd_epochs=30, pgd_lr=1e-5, pgd_weight_decay=1e-3,
                        train_eps=0.1, train_steps=10, train_step_size=0.01, train_restarts=2,
                        val_eps=0.1, val_steps=50, val_step_size=0.005, val_restarts=5,
                        eps_budgets=(0.05, 0.1, 0.15),
                        batch_size=32, fold_seed_offset=0,
                        save_artifacts=False, artifact_dir=None,
                        alignment_whiteners=None,
                        distill_student=False, student_epochs=20,
                        student_lr=5e-4, student_weight_decay=1e-4,
                        distill_temperature=4.0, distill_alpha=0.7,
                        distill_on_adversarial=False, student_kwargs=None,
                        export_student_torchscript=True):
    fold_seed = SEED + fold_seed_offset
    torch.manual_seed(fold_seed)
    np.random.seed(fold_seed)

    test_ids = _as_subject_list(test_subj)
    test_id_set = set(test_ids)
    remaining = [s for s in all_subj_ids if s not in test_id_set]
    if len(remaining) <= n_val_subjects:
        raise ValueError("n_val_subjects must leave at least one training subject")

    rng = random.Random(fold_seed)
    rng.shuffle(remaining)
    val_ids = sorted(remaining[:n_val_subjects])
    train_ids = sorted(remaining[n_val_subjects:])

    X_train, y_train, sid_train = build_split(train_ids, aligned_data, subject_labels)
    X_val, y_val, sid_val = build_split(val_ids, aligned_data, subject_labels)
    X_test, y_test, sid_test = build_split(test_ids, aligned_data, subject_labels)

    train_set = EEGMATAlignedDataset(X_train, y_train, sid_train)
    val_set = EEGMATAlignedDataset(X_val, y_val, sid_val)
    test_set = EEGMATAlignedDataset(X_test, y_test, sid_test)

    counts = torch.bincount(train_set.y, minlength=2).clamp_min(1)
    sample_weights = (1.0 / counts.float())[train_set.y].double()
    sampler = torch.utils.data.WeightedRandomSampler(
        sample_weights, num_samples=len(train_set), replacement=True
    )

    pin_memory = device.type == "cuda"
    train_loader = DataLoader(train_set, batch_size=batch_size, sampler=sampler, pin_memory=pin_memory)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)

    # ---- Stage 1: clean EEGNet, selected on validation balanced accuracy ----
    model = EEGNET(num_classes=2, C=C, T=T).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=clean_lr, weight_decay=clean_weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=clean_epochs)

    best_val_bal_acc = -1.0
    best_clean_state = None
    for epoch in range(1, clean_epochs + 1):
        train_one_epoch(model, train_loader, optimizer, criterion, device)
        _, val_bal_acc, _ = evaluate_balanced(model, val_loader, device)
        scheduler.step()
        if val_bal_acc > best_val_bal_acc:
            best_val_bal_acc = val_bal_acc
            best_clean_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    clean_model = EEGNET(num_classes=2, C=C, T=T).to(device)
    clean_model.load_state_dict(best_clean_state)
    clean_model.eval()

    # ---- Stage 2: plain PGD-Linf adversarial training from the clean checkpoint ----
    pgd_model = EEGNET(num_classes=2, C=C, T=T).to(device)
    pgd_model.load_state_dict(best_clean_state)

    train_step_size = 0.01 if train_step_size is None else train_step_size
    val_eps = train_eps if val_eps is None else val_eps
    val_step_size = 0.005 if val_step_size is None else val_step_size

    train_attacker = PGDLinfAttack(eps=train_eps, n_steps=train_steps, step_size=train_step_size,
                                   random_init=True, n_restarts=train_restarts)
    val_attacker = PGDLinfAttack(eps=val_eps, n_steps=val_steps, step_size=val_step_size,
                                 random_init=True, n_restarts=val_restarts,
                                 objective="cw", kappa=0.0)

    pgd_optimizer = torch.optim.Adam(pgd_model.parameters(), lr=pgd_lr,
                                     weight_decay=pgd_weight_decay)
    pgd_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(pgd_optimizer, T_max=pgd_epochs)
    pgd_criterion = nn.CrossEntropyLoss()

    best_val_rob_bal_acc = -1.0
    best_pgd_state = {k: v.detach().clone() for k, v in pgd_model.state_dict().items()}
    for epoch in range(1, pgd_epochs + 1):
        pgd_adversarial_train_one_epoch_balanced(
            pgd_model, train_loader, pgd_optimizer, train_attacker, device,
            criterion=pgd_criterion, num_classes=2,
        )
        val_rob_bal_acc = evaluate_adversarial_balanced(pgd_model, val_loader, val_attacker, device)
        pgd_scheduler.step()
        if val_rob_bal_acc > best_val_rob_bal_acc:
            best_val_rob_bal_acc = val_rob_bal_acc
            best_pgd_state = {k: v.detach().clone() for k, v in pgd_model.state_dict().items()}

    pgd_model.load_state_dict(best_pgd_state)
    pgd_model.eval()

    # ---- Optional Stage 2b: distill adversarial teacher into compact student ----
    student_model, student_history, best_student_val_bal_acc = None, None, None
    resolved_student_kwargs = default_student_kwargs() if student_kwargs is None else dict(student_kwargs)
    if distill_student:
        distill_attacker = train_attacker if distill_on_adversarial else None
        student_model, student_history, best_student_val_bal_acc = distill_student_from_teacher(
            pgd_model, train_loader, val_loader, device,
            student_kwargs=resolved_student_kwargs, epochs=student_epochs,
            lr=student_lr, weight_decay=student_weight_decay,
            temperature=distill_temperature, alpha=distill_alpha,
            adv_attacker=distill_attacker,
        )

    # ---- Stage 3: evaluate both models on the held-out subject group with PGD-CW ----
    models = {"Clean_EEGNet": clean_model, "PGD_AT_Linf": pgd_model}
    if student_model is not None:
        models["Student_KD"] = student_model
    fold_metrics = {}

    res_clean_eval = {name: evaluate_full(m, test_loader, device, attacker=None, compute_fidelity=False)
                      for name, m in models.items()}
    for name, res in res_clean_eval.items():
        m = compute_metrics(res)
        fold_metrics[(name, "clean")] = m

    for eps in eps_budgets:
        attacker = make_eval_linf_attacker(eps)
        for name, model_ in models.items():
            res = evaluate_full(model_, test_loader, device, attacker=attacker, compute_fidelity=True)
            m = compute_metrics(res)
            fold_metrics[(name, eps)] = m

    artifact_paths = {}
    if save_artifacts:
        artifact_dir = Path("./artifacts/eegmat_8ch_pgd_cw_kd") if artifact_dir is None else Path(artifact_dir)
        fold_extra = {
            "best_val_bal_acc": best_val_bal_acc,
            "best_val_rob_bal_acc": best_val_rob_bal_acc,
            "train_eps": train_eps,
            "train_steps": train_steps,
            "train_restarts": train_restarts,
            "val_eps": val_eps,
            "val_steps": val_steps,
            "val_restarts": val_restarts,
        }
        clean_name = make_fold_artifact_name(fold_seed_offset, test_ids, "clean")
        teacher_name = make_fold_artifact_name(fold_seed_offset, test_ids, "pgd_teacher")
        artifact_paths["clean"] = save_deployment_checkpoint(
            artifact_dir / f"{clean_name}.pt", clean_model, "Clean_EEGNet",
            default_teacher_kwargs(), train_ids, val_ids, test_ids,
            metrics={k: v for k, v in fold_metrics.items() if k[0] == "Clean_EEGNet"},
            alignment_whiteners=alignment_whiteners, extra=fold_extra,
        )
        artifact_paths["teacher"] = save_deployment_checkpoint(
            artifact_dir / f"{teacher_name}.pt", pgd_model, "PGD_AT_Linf",
            default_teacher_kwargs(), train_ids, val_ids, test_ids,
            metrics={k: v for k, v in fold_metrics.items() if k[0] == "PGD_AT_Linf"},
            alignment_whiteners=alignment_whiteners, extra=fold_extra,
        )
        if student_model is not None:
            student_name = make_fold_artifact_name(fold_seed_offset, test_ids, "student_kd")
            student_extra = {
                **fold_extra,
                "teacher_checkpoint": artifact_paths["teacher"],
                "best_student_val_bal_acc": best_student_val_bal_acc,
                "distill_temperature": distill_temperature,
                "distill_alpha": distill_alpha,
                "distill_on_adversarial": distill_on_adversarial,
                "student_parameters": count_parameters(student_model),
                "teacher_parameters": count_parameters(pgd_model),
            }
            artifact_paths["student"] = save_deployment_checkpoint(
                artifact_dir / f"{student_name}.pt", student_model, "Student_KD",
                resolved_student_kwargs, train_ids, val_ids, test_ids,
                metrics={k: v for k, v in fold_metrics.items() if k[0] == "Student_KD"},
                alignment_whiteners=alignment_whiteners,
                distillation_history=student_history,
                extra=student_extra,
            )
            if export_student_torchscript:
                artifact_paths["student_torchscript"] = export_torchscript_model(
                    student_model, artifact_dir / f"{student_name}.torchscript.pt", C=C, T=T
                )

    return {
        "test_subject": test_ids[0] if len(test_ids) == 1 else tuple(test_ids),
        "test_subjects": test_ids,
        "val_subjects": val_ids,
        "train_subjects": train_ids,
        "best_val_bal_acc": best_val_bal_acc,
        "best_val_rob_bal_acc": best_val_rob_bal_acc,
        "metrics": fold_metrics,
        "artifact_paths": artifact_paths,
        "student_history": student_history,
        "best_student_val_bal_acc": best_student_val_bal_acc,
    }


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

all_subj_ids = sorted(aligned_data.keys())

N_GROUPED_FOLDS = 6
TEST_FOLDS = make_subject_folds(all_subj_ids, n_folds=N_GROUPED_FOLDS, seed=SEED, shuffle=True)
covered_subjects = sorted(s for fold in TEST_FOLDS for s in fold)
assert len(TEST_FOLDS) == 6, "Expected exactly 6 grouped LOSO folds"
assert covered_subjects == all_subj_ids, "Grouped folds must cover every subject exactly once"

EPS_BUDGETS = (0.05, 0.1, 0.15)
TRAIN_EPS = 0.10
MAIN_EPS = TRAIN_EPS

ARTIFACT_DIR = Path("./artifacts/eegmat_8ch_pgd_cw_kd")
SAVE_ARTIFACTS = True
DISTILL_STUDENT = True
STUDENT_EPOCHS = 20
DISTILL_ON_ADVERSARIAL = False
EXPORT_STUDENT_TORCHSCRIPT = True

print(f"Running {len(TEST_FOLDS)} grouped held-out folds over {len(all_subj_ids)} subjects")
print("Held-out subject groups:", TEST_FOLDS)

loso_results = []
for i, test_subjects in enumerate(TEST_FOLDS):
    result = run_loso_fold_full(
        test_subj=test_subjects,
        all_subj_ids=all_subj_ids,
        aligned_data=aligned_data,
        subject_labels=subject_labels,
        device=device,
        n_val_subjects=4,
        clean_epochs=30,
        pgd_epochs=30,
        train_eps=TRAIN_EPS,
        train_steps=10,
        train_step_size=0.01,
        train_restarts=2,
        val_eps=MAIN_EPS,
        val_steps=50,
        val_step_size=0.005,
        val_restarts=5,
        eps_budgets=EPS_BUDGETS,
        fold_seed_offset=i,
        save_artifacts=SAVE_ARTIFACTS,
        artifact_dir=ARTIFACT_DIR,
        alignment_whiteners=alignment_whiteners,
        distill_student=DISTILL_STUDENT,
        student_epochs=STUDENT_EPOCHS,
        distill_on_adversarial=DISTILL_ON_ADVERSARIAL,
        export_student_torchscript=EXPORT_STUDENT_TORCHSCRIPT,
    )
    loso_results.append(result)
    m_clean = result["metrics"][("Clean_EEGNet", "clean")]
    m_pgd_main = result["metrics"][("PGD_AT_Linf", MAIN_EPS)]
    msg = (
        f"[fold {i+1:>2}/{len(TEST_FOLDS)}] held-out subjects={test_subjects}  "
        f"clean_bal_acc={m_clean['bACC_clean']:.4f}  "
        f"pgd_rob_bal_acc@eps={MAIN_EPS}={m_pgd_main['bACC_rob']:.4f}"
    )
    if ("Student_KD", MAIN_EPS) in result["metrics"]:
        m_student_main = result["metrics"][("Student_KD", MAIN_EPS)]
        msg += f"  student_rob_bal_acc@eps={MAIN_EPS}={m_student_main['bACC_rob']:.4f}"
    if result.get("artifact_paths"):
        msg += f"  artifacts={result['artifact_paths']}"
    print(msg)


In [ ]:
def mean_ci(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    n = len(values)
    mean = values.mean()
    if n < 2:
        return mean, mean, mean
    sem = values.std(ddof=1) / np.sqrt(n)
    tcrit = stats.t.ppf(1 - alpha / 2, df=n - 1)
    return mean, mean - tcrit * sem, mean + tcrit * sem


SUMMARY_MODELS = ["Clean_EEGNet", "PGD_AT_Linf"]
if any(("Student_KD", "clean") in r["metrics"] for r in loso_results):
    SUMMARY_MODELS.append("Student_KD")

print("=" * 100)
print("Clean test performance, aggregated across grouped held-out folds")
print("=" * 100)
for model_name in SUMMARY_MODELS:
    vals = [r["metrics"][(model_name, "clean")]["bACC_clean"] for r in loso_results]
    mean, lo, hi = mean_ci(vals)
    print(f"{model_name:<18} bACC_clean: mean={mean:.4f}  min={min(vals):.4f}  max={max(vals):.4f}  95% CI=({lo:.4f}, {hi:.4f})")

print()
print("=" * 100)
print("PGD-CW robust test performance per epsilon, aggregated across grouped held-out folds")
print("=" * 100)
for model_name in SUMMARY_MODELS:
    print(f"\nModel: {model_name}")
    for eps in EPS_BUDGETS:
        a_rob = [r["metrics"][(model_name, eps)]["A_rob"] for r in loso_results]
        bacc_rob = [r["metrics"][(model_name, eps)]["bACC_rob"] for r in loso_results]
        mean_a, lo_a, hi_a = mean_ci(a_rob)
        mean_b, lo_b, hi_b = mean_ci(bacc_rob)
        print(
            f"  eps={eps:.3f}  A_rob mean={mean_a:.4f} CI=({lo_a:.4f},{hi_a:.4f})   "
            f"bACC_rob mean={mean_b:.4f} CI=({lo_b:.4f},{hi_b:.4f})"
        )


In [ ]:
# Paired Wilcoxon: Clean_EEGNet vs PGD_AT_Linf robust balanced accuracy
# at the main training epsilon. Each paired value is one grouped held-out fold,
# so this grouped-fold setup tests paired fold-level metrics, not per-subject metrics.
clean_vals = np.array([r["metrics"][("Clean_EEGNet", MAIN_EPS)]["bACC_rob"] for r in loso_results])
pgd_vals = np.array([r["metrics"][("PGD_AT_Linf", MAIN_EPS)]["bACC_rob"] for r in loso_results])
heldout_groups = [r.get("test_subjects", [r["test_subject"]]) for r in loso_results]
n_heldout_subjects = sum(len(group) for group in heldout_groups)

print("=" * 100)
print(
    f"Paired Wilcoxon signed-rank test across {len(heldout_groups)} grouped "
    f"held-out folds ({n_heldout_subjects} subjects total), eps={MAIN_EPS}"
)
print("H0: no paired difference between Clean_EEGNet and PGD_AT_Linf robust balanced accuracy")
print("=" * 100)
print(f"Mean robust bACC ? Clean_EEGNet : {clean_vals.mean():.4f}")
print(f"Mean robust bACC ? PGD_AT_Linf  : {pgd_vals.mean():.4f}")

if np.allclose(pgd_vals - clean_vals, 0):
    print("All per-fold differences are zero ? Wilcoxon cannot be computed.")
else:
    stat, pval = wilcoxon(clean_vals, pgd_vals, alternative="two-sided")
    print(f"Wilcoxon statistic: {stat:.4f}")
    print(f"p-value           : {pval:.4f}")
    print("Result:", "statistically significant at p < 0.05" if pval < 0.05 else "not statistically significant at p = 0.05")


## Notes on this version vs. the Stackelberg notebook

- EDF files are trimmed to the 8 selected 10-20 electrodes F3, F4, Fz, P3, P4, Pz, O1, and O2 before epoching, EA alignment, splitting, and model construction.
- The adversarial training stage is still plain PGD adversarial training, not Stackelberg training. There is no differentiable follower, feature matching term, or clean/adversarial leader weighting.
- Training PGD uses cross-entropy loss, random initialization in the L-infinity ball, sign-gradient ascent, and projection back into the L-infinity ball after each step.
- Validation and final robust evaluation now use PGD-CW loss: the untargeted CW margin `best_other_logit - true_class_logit` is maximized during PGD and used for vectorized best-restart selection.
- Shared experimental hyperparameters are matched to the Stackelberg notebook for fair comparison: `train_eps=0.10`, `EPS_BUDGETS=(0.05, 0.1, 0.15)`, train PGD uses 10 steps, step size `0.01`, and 2 restarts; validation PGD-CW uses 50 steps, step size `0.005`, and 5 restarts; final robust evaluation uses 50 steps, step size `epsilon / 10`, and 5 restarts.
- LOSO is grouped into exactly 6 held-out folds. With all 36 EEGMAT subjects loaded, each fold holds out 6 subjects and every subject appears in one held-out group.
- The dataset object stores tensorized samples once, and the attack implementation vectorizes random restarts across the batch dimension for both CE and CW objectives.
- `SAVE_ARTIFACTS=True` saves fold-level clean and adversarial teacher checkpoints with selected-channel metadata, per-subject causal-EA whiteners, fold splits, and metrics.
- `DISTILL_STUDENT=True` trains a compact EEGNet student (`F1=4`, `D=1`, `F2=8`) from the adversarial teacher using temperature-scaled KL plus hard-label CE, then saves both a PyTorch checkpoint and TorchScript export for deployment.
